# 手撕Self-Attention

**导入所需包**

In [1]:
import torch
import torch.nn as nn
import torch.functional as F
import math

/home/panjiazhou/envs/pytorch/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**生成测试数据和自注意力机制参数，输出维度d_v，维度d_model**

In [3]:
X = torch.randn(128, 64, 512) # batch, seq_len(time), dim
batch, time, d_model = X.shape
d_model = d_model
d_v = 512 
d_k = 512


NameError: name 'torch' is not defined

**编写自注意力机制代码**

In [3]:
class self_attention(nn.Module):
    def __init__(self, d_model, d_k, d_v, dropout=0.1):
        super(self_attention, self).__init__()
        '''
        d_model 输入数据的特征维度
        d_k wq和wk的输出维度
        d_v 最终的输出维度
        '''
        self.d_model = d_model
        self.dropout = nn.Dropout(p=dropout)
        self.w_q = nn.Linear(d_model, d_k)
        self.w_k = nn.Linear(d_model, d_k)
        self.w_v = nn.Linear(d_model, d_v)
        self.softmax = nn.Softmax(dim=-1)
        self.d_k = d_k
    def forward(self, x, mask=None):
        batch, time, dim = x.shape
        q = self.w_q(x)
        k = self.w_k(x)
        v = self.w_v(x)

        score = self.softmax(q @ k.transpose(-2,-1) / math.sqrt(self.d_k))

        score = self.dropout(score)
        if mask is not None:
            mask = torch.tril(time, time)
            score = torch.masked_fill(mask==0,"-inf")
        
        score = score @ v
        return score

In [4]:
import torch.nn as nn
import torch.functional as F
import math
import torch

class SA(nn.Module):
    def __init__(self,d_model, dim_k,dim_v,dropout=0.1,mask=None):
        super(SA,self).__init__()
        self.w_q = nn.Linear(d_model,dim_k)
        self.w_k = nn.Linear(d_model,dim_k)
        self.w_v = nn.Linear(d_model,dim_v)
        self.d_model = d_model
        self.mask = mask
        self.dim_k = dim_k
        self.dim_v = dim_v
        self.softmax = nn.Softmax(dim=-1)
        if dropout>0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = nn.Identity()
    def forward(self, x):
        q,k,v = self.w_q(x),self.w_k(x),self.w_v(x)
        score = q @ k.transpose(-2,-1) / math.sqrt(self.dim_k)
        self.dropout(score)
        if self.mask:
            mask_trill = torch.tril(torch.ones(d_model,d_model),dtype=bool)
            mask_score = score.masked_fill(mask_trill==0,float('-inf'))
            final_score = self.softmax(mask_score) @ v
        else:
            final_score = self.softmax(score) @ v
        return score
        

**测试结果**

In [5]:
self_att = SA(d_model, d_k, d_v)
output = self_att(X)
print(output, output.shape)

tensor([[[-0.1356, -0.2601, -0.1526,  ..., -0.4884,  0.0304, -0.0571],
         [-0.0986, -0.4886, -0.4538,  ..., -0.1908,  0.2821, -0.0273],
         [ 0.0942,  0.2571, -0.1461,  ...,  0.8046,  0.1518, -0.0574],
         ...,
         [ 0.1093, -0.0031,  0.1018,  ...,  0.1511,  0.1856, -0.0736],
         [ 0.0993, -0.3842,  0.2606,  ...,  0.1711,  0.3650,  0.1857],
         [-0.1746,  0.4824, -0.3225,  ...,  0.7716, -0.2573,  0.8183]],

        [[ 0.0403,  0.0257,  0.1422,  ..., -0.0393,  0.0218,  0.2488],
         [ 0.0709, -0.0339, -0.2216,  ...,  0.2373, -0.3923, -0.0344],
         [-0.1971, -0.4005,  0.4002,  ..., -0.0503,  0.7116,  0.3521],
         ...,
         [-0.1893,  0.1585, -0.2484,  ...,  0.2013,  0.2537, -0.1616],
         [-0.4479, -0.4351, -0.1575,  ...,  0.1497, -0.0300,  0.5367],
         [ 0.2141, -0.2003,  0.2627,  ..., -0.2400,  0.2032,  0.5023]],

        [[-0.2278, -0.4517,  0.1388,  ...,  0.4084, -0.1250, -0.0547],
         [ 0.1903,  0.3431, -0.3944,  ..., -0

# 手撕Cross Attention

In [ ]:
import torch.nn as nn
import torch.functional as F
import math
import torch

class cross_att(nn.Module):
    def __init__(self,d_model, dim_k,dim_v,dropout=0.1,mask=None):
        super(cross_att,self).__init__()
        self.w_q = nn.Linear(d_model,dim_k)
        self.w_k = nn.Linear(d_model,dim_k)
        self.w_v = nn.Linear(d_model,dim_v)
        self.d_model = d_model
        self.mask = mask
        self.dim_k = dim_k
        self.dim_v = dim_v
        self.softmax = nn.Softmax(dim=-1)
        if dropout>0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = nn.Identity()
    def forward(self, encoder_input,decoder_input):
        q,k,v = self.w_q(encoder_input),self.w_k(decoder_input),self.w_v(decoder_input)
        score = q @ k.transpose(-2,-1) / math.sqrt(self.dim_k)
        self.dropout(score)
        if self.mask:
            mask_trill = torch.tril(torch.ones(d_model,d_model),dtype=bool)
            mask_score = score.masked_fill(mask_trill==0,float('-inf'))
            final_score = self.softmax(mask_score) @ v
        else:
            final_score = self.softmax(score) @ v
        return score
        

# 手撕Multi-head attention

**编辑多头注意力机制参数,生成测试数据和多头注意力的头数及维度$d_k$**

In [6]:
X = Y = Z = torch.randn(256, 128, 512)
batch, time, dim = X.shape
d_model = dim
n_head = 8

**编写多头注意力机制代码**

In [7]:
class multi_head_attention(nn.Module):
    def __init__(self, d_model, n_head, bias=None, dropout=0.1):
        super(multi_head_attention, self).__init__()

        self.d_model = d_model
        self.n_head = n_head
        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)
        self.bias = bias
        self.combine = nn.Linear(d_model,d_model)
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(p=dropout)
    def f_view(self, x):
        batch, time, dim = x.shape
        n_d = self.d_model // self.n_head
        return x.view(batch, time, self.n_head, n_d).permute(0, 2, 1, 3)
    def forward(self, q, k, v, mask=None):
        batch, time ,dim = q.shape
        n_d = self.d_model // self. n_head
        
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)
        q = self.f_view(q)
        k = self.f_view(k)
        v = self.f_view(v)
        score = q @ k.transpose(2,3)/math.sqrt(n_d)
        score = self.dropout(score)
        if mask is not None:
            mask = torch.tril(torch.ones(time,time,dtype=bool))
            mask_score = score.masked_fill(mask==0, float("-inf"))        

            final_score = self.softmax(mask_score) @ v
        else:
            final_score = self.softmax(score) @ v
        final_score = final_score.permute(0,2,1,3).contiguous().view(batch, time, d_model)
        final_score = self.combine(final_score)

        return final_score

In [ ]:
import torch.nn as nn
import torch
import torch.functional as F
import math

class MHA(nn.Module):
    def __init__(self, d_model,head_n,dropout=0.1):
        super(MHA,self).__init__()
        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)
        self.w_o = nn.Linear(d_model,d_model)
        self.d_model = d_model
        self.head_n = head_n
        
        self.softmax = nn.Softmax(dim=-1)
        if dropout>0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = nn.Identity()
    def f_view(self,x):
        batch,seq,dim = x.shape
        dim_n = self.d_model//self.head_n
        return x.view(batch,seq,self.head_n,dim_n).permute(0,2,1,3)
    
    def forward(self,q,k,v,mask=None):
        batch,seq,dim = q.shape
        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)
        
        q = self.f_view(q)
        k = self.f_view(k)
        v = self.f_view(v)

        score = q @ k.transpose(2,3)/math.sqrt(self.d_model//self.head_n)
        score = self.dropout(score)
        if mask:
            mask_trill = torch.tril(torch.ones(seq,seq),dtype=bool)
            mask_score = torch.masked(mask_trill==0,float('-inf'))
            final_score = self.softmax(mask_score) @ v
        else:
            final_score = self.softmax(score)@v
        
        final_score = final_score.permute(0,2,1,3).contiguous().view(batch,seq,dim)
        final_score = self.w_o(final_score)
        return final_score
        
        


**测试结果**

In [9]:
mha = MHA(d_model, n_head)
output = mha(X,Y,Z)
print(output, output.shape)

tensor([[[-0.0016,  0.0737,  0.0781,  ..., -0.0424, -0.0303,  0.0508],
         [ 0.0183,  0.0615,  0.0589,  ..., -0.0482, -0.0388,  0.0515],
         [ 0.0152,  0.0532,  0.0809,  ..., -0.0350, -0.0422,  0.0687],
         ...,
         [ 0.0207,  0.0787,  0.0721,  ..., -0.0339, -0.0232,  0.0592],
         [ 0.0158,  0.0563,  0.0542,  ..., -0.0566, -0.0299,  0.0489],
         [ 0.0046,  0.0721,  0.0821,  ..., -0.0301, -0.0410,  0.0298]],

        [[-0.0327, -0.0234,  0.0145,  ...,  0.0081, -0.0294,  0.0325],
         [-0.0302, -0.0111,  0.0252,  ..., -0.0020, -0.0462,  0.0350],
         [-0.0430, -0.0272,  0.0119,  ...,  0.0147, -0.0410,  0.0039],
         ...,
         [-0.0392, -0.0228,  0.0193,  ..., -0.0097, -0.0319,  0.0020],
         [-0.0182, -0.0351,  0.0441,  ..., -0.0175, -0.0421,  0.0418],
         [-0.0292, -0.0305,  0.0251,  ..., -0.0229, -0.0263,  0.0092]],

        [[ 0.0647,  0.0984,  0.0461,  ..., -0.0415,  0.0018,  0.0224],
         [ 0.0329,  0.0606,  0.0488,  ..., -0